In [1]:
import sys, os
sys.path.append(os.path.abspath("../../"))
from src.dataset.data_loader import load_competition_data

train_dataset, test_dataset, sample_submission = load_competition_data()
print("Train shape:", train_dataset.shape)
print("Test shape :", test_dataset.shape)

Train shape: (2000, 7)
Test shape : (500, 7)


In [2]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import torch
import numpy as np
from transformers import (
    AutoTokenizer
)

c:\Users\ais\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ===========================================================================
# setup — build KB(knowledge base) + FAISS index from train.csv correct answers
# ============================================================================.
train_df_m3 = train_dataset.copy()

print("Creating knowledge base")
kb = []
for idx, row in train_df_m3.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))   # pull out the correct option text & add it to knowledge base

print("Loading embedding model and creating index")
m3_model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = m3_model.encode(kb, show_progress_bar=False)    # (2000, 284)
m3_index = faiss.IndexFlatL2(kb_embeddings.shape[1])   # fiass index - structure to store the embedds for fast query
m3_index.add(kb_embeddings)
print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3651.44it/s]


Knowledge base successfully created


In [4]:
# ============================================================
# Zero-shot classifier setup for Q1, Q2, Q6
# ============================================================
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

row_150 = train_dataset.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [
    str(row_150['A']), 
    str(row_150['B']), 
    str(row_150['C']), 
    str(row_150['D']), 
    str(row_150['E'])
]
ans_150 = str(row_150[row_150['answer']])

print(" ")
print("Prompt (row 150) :", prompt_150)
# print("Labels  (A-E)    :", labels_150)
print(" ")
print("Ground truth text (correct option):", ans_150)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 5190.78it/s]


 
Prompt (row 150) : Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.
 
Ground truth text (correct option): The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."


In [5]:
# ============================================================
# Q1 — Zero-shot probability assigned to the ground-truth option, row 150
# ============================================================
result_q1 = zs(prompt_150, labels_150, multi_label=False)

label_to_score = dict(zip(result_q1['labels'], result_q1['scores']))
q1_answer = round(label_to_score[ans_150], 3)
print(f"\nQ1 answer — probability of ground-truth option: {q1_answer}")
print(" ")

# ============================================================
# Q2 — FAISS rank of the true document (KB index 150) for the row-150 prompt, k=10
# ============================================================
query_vec_150 = m3_model.encode([prompt_150], show_progress_bar=False)
k = 10
distances, retrieved_indices = m3_index.search(np.array(query_vec_150).astype('float32'), k)
retrieved_indices = retrieved_indices[0].tolist()

print(f"Top-{k} retrieved KB indices : {retrieved_indices}")
if 150 in retrieved_indices:
    q2_answer = retrieved_indices.index(150) + 1   # convert 0-based position -> 1-based rank
    print(f"\nQ2 answer — FAISS rank of true document (KB idx 150): {q2_answer}")
else:
    q2_answer = None
    print(f"\nQ2 answer — true document (KB idx 150) was NOT in the top-{k}")


Q1 answer — probability of ground-truth option: 0.384
 
Top-10 retrieved KB indices : [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]

Q2 answer — FAISS rank of true document (KB idx 150): 10


In [6]:
# ==================================================================
# Q3 — Cross-encoder reranking of the top-10 FAISS docs for row 150
# =================================================================
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices]        # top-10 chunks retrieved by FAISS above
pairs = [[prompt_150, doc] for doc in docs_10]      # prompt-retrieved-context pairs
ce_scores = cross_encoder.predict(pairs)            # cross-encoder relevance score per pair

ranked_order = np.argsort(ce_scores)[::-1]          # highest score first
ranked_kb_indices = [retrieved_indices[i] for i in ranked_order]

print("Cross-encoder ranking (rank : KB idx : score):")
for rank, i in enumerate(ranked_order, start=1):
    kb_idx = retrieved_indices[i]
    marker = "  <-- true document" if kb_idx == 150 else ""
    print(f"  rank {rank:2d}  kb_idx {kb_idx:4d}  score {ce_scores[i]:.4f}{marker}")

if 150 in ranked_kb_indices:
    q3_answer = ranked_kb_indices.index(150) + 1
    print(f"\nQ3 answer — Cross-Encoder rank of true document: {q3_answer}")
else:
    q3_answer = None
    print("\nQ3 answer — true document not among the reranked candidates")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4224.80it/s]


Cross-encoder ranking (rank : KB idx : score):
  rank  1  kb_idx  150  score 4.7585  <-- true document
  rank  2  kb_idx 1906  score 4.7526
  rank  3  kb_idx  847  score 4.7526
  rank  4  kb_idx 1693  score 4.7526
  rank  5  kb_idx 1269  score 4.7375
  rank  6  kb_idx 1532  score 4.7375
  rank  7  kb_idx  168  score 4.7072
  rank  8  kb_idx  576  score 4.6870
  rank  9  kb_idx 1701  score 4.6602
  rank 10  kb_idx  663  score 4.6602

Q3 answer — Cross-Encoder rank of true document: 1


In [7]:
# ============================================================
# Q4 — Token count of the RAG string for row 42 (k=5 docs, bert-base-uncased, no truncation)
# ============================================================
bert_uncased_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

row_42 = train_dataset.iloc[42]
prompt_42 = str(row_42['prompt'])

query_vec_42 = m3_model.encode([prompt_42], show_progress_bar=False)
_, idx_42 = m3_index.search(np.array(query_vec_42).astype('float32'), 5)
docs_42 = [kb[i] for i in idx_42[0]]

concatenated_docs_42 = " ".join(docs_42)
rag_string_42 = f"Context: {concatenated_docs_42} Question: {prompt_42}"

tokens_42 = bert_uncased_tokenizer(rag_string_42, truncation=False)['input_ids']
q4_answer = len(tokens_42)

print("RAG string (row 42):")
print(rag_string_42)
print(f"\nQ4 answer — total token count (bert-base-uncased, no truncation): {q4_answer}")

RAG string (row 42):
Context: Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to t

In [8]:
# =====================================================================
# Q5 — Zero-shot probability using the TRUE document as context, row 150
# ====================================================================
true_document_150 = kb[150]
rag_string_150 = f"Context: {true_document_150} Question: {prompt_150}"

result_q5 = zs(rag_string_150, labels_150, multi_label=False)
label_to_score_q5 = dict(zip(result_q5['labels'], result_q5['scores']))
q5_answer = round(label_to_score_q5[ans_150], 3)

print("RAG string (row 150, true document):")
print(rag_string_150)
print("\nZero-shot scores with true-document context:")
for lbl, sc in zip(result_q5['labels'], result_q5['scores']):
    marker = "  <-- ground truth" if lbl == ans_150 else ""
    print(f"  {sc:.3f}  {lbl}{marker}")

print(f"\nQ5 answer — probability of ground-truth option with true-document RAG: {q5_answer}")

RAG string (row 150, true document):
Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.

Zero-shot scores with true-document context:
  0.989  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."  <-- ground truth
  0.004  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
  0.003  The butterfly effect is the pheno

In [9]:
# ============================================================================
# Q6 — Adversarial RAG: force context = unrelated KB doc at index 999, row 150
# ==========================================================================
adversarial_document = kb[999]
adversarial_rag_string = f"Context: {adversarial_document} Question: {prompt_150}"

result_q6 = zs(adversarial_rag_string, labels_150, multi_label=False)
label_to_score_q6 = dict(zip(result_q6['labels'], result_q6['scores']))
q6_answer = round(label_to_score_q6[ans_150], 3)

print("Adversarial context (KB idx 999):", adversarial_document)
print("\nZero-shot scores with adversarial context:")
for lbl, sc in zip(result_q6['labels'], result_q6['scores']):
    marker = "  <-- ground truth" if lbl == ans_150 else ""
    print(f"  {sc:.3f}  {lbl}{marker}")

print(f"\nQ6 answer — probability of ground-truth option with adversarial RAG: {q6_answer}")

Adversarial context (KB idx 999): A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics.

Zero-shot scores with adversarial context:
  0.529  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."  <-- ground truth
  0.425  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
  0.020  The butterfly effect is the phenomenon t

In [10]:
# ============================================================
# Q7 — Hit Rate @5 for the first 100 rows (indices 0-99)
# A 'hit' = the exact correct-option string appears inside any of the top-5 retrieved docs
# ====================================================================================
hits = 0
n_rows_q7 = 100

for i in range(n_rows_q7):
    row = train_dataset.iloc[i]
    prompt_i = str(row['prompt'])
    correct_text = str(row[row['answer']])

    q_vec = m3_model.encode([prompt_i], show_progress_bar=False)
    _, idx_i = m3_index.search(np.array(q_vec).astype('float32'), 5)
    retrieved_docs_i = [kb[j] for j in idx_i[0]]

    if any(correct_text in doc for doc in retrieved_docs_i):
        hits += 1

q7_answer = round(hits / n_rows_q7 * 100, 1)
print(f"Hits: {hits} / {n_rows_q7}")
print(f"Q7 answer — Hit Rate @5 (%): {q7_answer}")

Hits: 73 / 100
Q7 answer — Hit Rate @5 (%): 73.0


In [11]:
# ============================================================
# Q8 — Full RAG pipeline (Retrieve -> Rerank -> Augment -> Predict -> Score), rows 0-19
# ============================================================
OPTIONS = ['A', 'B', 'C', 'D', 'E']

def apk_letter(actual, predicted_letters, k=3):
    predicted_letters = predicted_letters[:k]
    if actual not in predicted_letters:
        return 0.0
    return 1.0 / (predicted_letters.index(actual) + 1)

ap_scores_q8 = []

for i in range(20):
    row = train_dataset.iloc[i]
    prompt_i = str(row['prompt'])
    correct_letter = row['answer']

    # ---- Retrieve: top-5 docs from FAISS ----
    q_vec = m3_model.encode([prompt_i], show_progress_bar=False)
    _, idx_i = m3_index.search(np.array(q_vec).astype('float32'), 5)
    retrieved_docs_i = [kb[j] for j in idx_i[0]]

    # ---- Rerank: cross-encoder, keep the single best document ----
    pairs_i = [[prompt_i, doc] for doc in retrieved_docs_i]
    scores_i = cross_encoder.predict(pairs_i)
    best_doc_i = retrieved_docs_i[int(np.argmax(scores_i))]

    # ---- Augment ----
    rag_string_i = f"Context: {best_doc_i} Question: {prompt_i}"

    # ---- Predict ----
    labels_i = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    result_i = zs(rag_string_i, labels_i, multi_label=False)

    # ---- Score: rank options highest -> lowest probability, map label text back to A-E ----
    label_to_letter_i = {str(row[opt]): opt for opt in OPTIONS}
    ranked_letters_i = [label_to_letter_i[lbl] for lbl in result_i['labels']]
    top3_letters_i = ranked_letters_i[:3]

    ap = apk_letter(correct_letter, top3_letters_i, k=3)
    ap_scores_q8.append(ap)

    print(f"row {i:2d} | true={correct_letter} | top3={top3_letters_i} | AP@3={ap:.3f}")

q8_answer = round(float(np.mean(ap_scores_q8)), 3)
print(f"\nQ8 answer — Final MAP@3 across 20 rows: {q8_answer}")

row  0 | true=B | top3=['B', 'D', 'A'] | AP@3=1.000
row  1 | true=A | top3=['A', 'E', 'C'] | AP@3=1.000
row  2 | true=C | top3=['C', 'D', 'B'] | AP@3=1.000
row  3 | true=B | top3=['B', 'D', 'A'] | AP@3=1.000
row  4 | true=A | top3=['A', 'B', 'C'] | AP@3=1.000
row  5 | true=C | top3=['B', 'C', 'A'] | AP@3=0.500
row  6 | true=E | top3=['E', 'B', 'D'] | AP@3=1.000
row  7 | true=A | top3=['A', 'B', 'C'] | AP@3=1.000
row  8 | true=A | top3=['A', 'C', 'D'] | AP@3=1.000
row  9 | true=A | top3=['A', 'B', 'C'] | AP@3=1.000
row 10 | true=C | top3=['C', 'A', 'D'] | AP@3=1.000
row 11 | true=B | top3=['B', 'C', 'A'] | AP@3=1.000
row 12 | true=D | top3=['D', 'A', 'B'] | AP@3=1.000
row 13 | true=E | top3=['E', 'D', 'B'] | AP@3=1.000
row 14 | true=E | top3=['E', 'A', 'D'] | AP@3=1.000
row 15 | true=E | top3=['E', 'B', 'C'] | AP@3=1.000
row 16 | true=C | top3=['C', 'B', 'E'] | AP@3=1.000
row 17 | true=C | top3=['C', 'B', 'E'] | AP@3=1.000
row 18 | true=B | top3=['B', 'D', 'A'] | AP@3=1.000
row 19 | tru

In [12]:
# ============================================================
# MILESTONE 3 Q/A — ANSWERS SUMMARY
# ============================================================
print("=== MILESTONE 3 Q/A — ANSWERS SUMMARY ===")
print(f"Q1 : {q1_answer}")
print(f"Q2 : {q2_answer}")
print(f"Q3 : {q3_answer}")
print(f"Q4 : {q4_answer}")
print(f"Q5 : {q5_answer}")
print(f"Q6 : {q6_answer}")
print(f"Q7 : {q7_answer}")
print(f"Q8 : {q8_answer}")

=== MILESTONE 3 Q/A — ANSWERS SUMMARY ===
Q1 : 0.384
Q2 : 10
Q3 : 1
Q4 : 216
Q5 : 0.989
Q6 : 0.529
Q7 : 73.0
Q8 : 0.975
